# SST Next-Brick Pose Prediction — COG-Aware Staged Model

Instead of predicting global (x, y) directly, the model makes a staged structural decision:

```
history
  ↓
Transformer encoder
  ↓
predict b          (binary: laying / standing)
  ↓
predict layer_id   (multi-class classification)
  ↓
predict support_state  (0=ground / 1=one-support / 2=two-support)
  ↓
select support-1 brick (cross-attention score over history tokens)
  ↓
select support-2 brick (optional, conditioned on s1)
  ↓
predict critical-point projections [alpha_A, perp_A, alpha_B, perp_B]
  ↓
recover brick center (x, y, r) from projections onto support critical lines
  ↓
snap z from layer_id lookup
```

**Structural critical points:** midpoints of the two *shortest* sides of the brick XY footprint.
**Projection:** alpha = normalised position along support's critical line; perp = normalised perpendicular offset.
All projection values are **SE(2)-invariant**, so augmented pairs reuse the same labels.

**Data:** batch2 and batch3 validated_simPhysics sequences

In [ ]:
import json
import os
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

cwd = Path(os.getcwd())
if cwd.name == "training_data":
    os.chdir(cwd.parent)
print(f"Working directory: {os.getcwd()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  PyTorch: {torch.__version__}")

In [ ]:
# Data
BATCH_DIRS = [
    "training_data/batch2/validated_simPhysics",
    "training_data/batch3/validated_simPhysics",
]
SEQ_SUBPATH = "5d_sequence/sequence.json"

# Model
FEATURE_DIM      = 17    # base(7)+layer_state(3)+same_layer(5)+geom(1)+sup_ctx(1)
PROJ_DIM         = 4     # [alpha_A, perp_A, alpha_B, perp_B]
N_SUPPORT_STATES = 3     # 0=ground/no-support, 1=one-support, 2=two-support
HIDDEN_DIM       = 128
N_HEADS          = 4
N_LAYERS         = 2
FF_DIM           = 256
DROPOUT          = 0.1
MAX_BRICKS       = 60
MAX_LAYER_NORM   = 20

# Brick geometry (physical dimensions in metres)
BRICK_W = 0.051   # long dimension when laying (= height when standing)
BRICK_D = 0.023   # short width (= long XY dimension when standing)
BRICK_H = 0.014   # height when lying flat

# Training
BATCH_SIZE   = 64
LR           = 3e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 200
PATIENCE     = 25
GRAD_CLIP    = 1.0
N_AUG        = 50

LAMBDA_B     = 1.0
LAMBDA_LAYER = 1.0
LAMBDA_SS    = 1.0
LAMBDA_S1    = 1.0
LAMBDA_S2    = 1.0
LAMBDA_PROJ  = 2.0

# Inference sampling noise (per projection component)
SIGMA_PROJ_INFER = [0.15, 0.08, 0.15, 0.08]

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Data Loading

In [ ]:
def load_sequences(batch_dirs, seq_subpath=SEQ_SUBPATH):
    sequences = []
    for bd in batch_dirs:
        bd_path = Path(bd)
        if not bd_path.exists():
            print(f"Warning: {bd_path} not found, skipping.")
            continue
        for demo_dir in sorted(bd_path.iterdir()):
            seq_path = demo_dir / seq_subpath
            if seq_path.exists():
                with open(seq_path) as f:
                    poses = json.load(f)
                sequences.append({"id": str(demo_dir), "poses": poses})
    return sequences


sequences = load_sequences(BATCH_DIRS)
print(f"Loaded {len(sequences)} sequences:\n")
for s in sequences:
    print(f"  {s['id']}  ->  {len(s['poses'])} bricks")

## 2. Feature Engineering

In [ ]:
def assign_layer_ids(poses, z_tol=1e-4):
    """Group bricks into layers by clustering similar z values."""
    unique_z = []
    for p in poses:
        z = p[2]
        if not any(abs(z - uz) < z_tol for uz in unique_z):
            unique_z.append(z)
    unique_z.sort()
    return [
        min(range(len(unique_z)), key=lambda i: abs(unique_z[i] - p[2]))
        for p in poses
    ]


def canonicalize_r(r):
    """Map yaw r to [0, pi) exploiting brick 180-degree symmetry."""
    r = r % math.pi
    return r + math.pi if r < 0 else r


# ── Structural critical points ────────────────────────────────────────────────

def get_critical_points(pose_5d):
    """
    Midpoints of the two shortest sides of the brick XY footprint in world XY.

    b=0 (laying):  footprint BRICK_W × BRICK_D; short sides at ±BRICK_W/2 along local x.
    b=1 (standing): footprint BRICK_D × BRICK_H; short sides at ±BRICK_D/2 along local x.

    Returns (cA, cB, half_span).
    """
    x, y, z, b, r = float(pose_5d[0]), float(pose_5d[1]), float(pose_5d[2]), int(pose_5d[3]), float(pose_5d[4])
    half_span = (BRICK_W if b == 0 else BRICK_D) / 2.0
    c, s = math.cos(r), math.sin(r)
    cA = [x + c * half_span, y + s * half_span]
    cB = [x - c * half_span, y - s * half_span]
    return cA, cB, half_span


def project_crit_to_support(tP, support_pose):
    """
    Project target critical point tP onto support brick's critical line.

    Returns (alpha, perp) both normalised by the support span (SE(2)-invariant):
      alpha ∈ [0, 1]  →  tP is within the support span
      perp  ≈ 0       →  tP lies on the support axis
    """
    sA, sB, s_half_span = get_critical_points(support_pose)
    span = 2.0 * s_half_span + 1e-8
    dx, dy = sB[0] - sA[0], sB[1] - sA[1]
    d  = math.sqrt(dx * dx + dy * dy + 1e-8)
    ax, ay = dx / d, dy / d   # unit axis from sA to sB
    px, py = -ay, ax           # 90° CCW perpendicular
    tdx, tdy = tP[0] - sA[0], tP[1] - sA[1]
    alpha = (ax * tdx + ay * tdy) / span
    perp  = (px * tdx + py * tdy) / span
    return alpha, perp


def compute_support_labels(target_pose, history_5d, history_layer_ids, target_layer_id):
    """
    Compute support state, support brick indices, and projection targets.

    Assigns each target critical point to the nearest support brick (min |perp|).
    Both map to same brick  →  one-support  (state=1).
    Different bricks        →  two-support  (state=2).
    No support bricks       →  ground       (state=0).

    All returned values are SE(2)-invariant — augmented pairs reuse them unchanged.

    Returns:
      support_state : int  (0, 1, or 2)
      s1_idx        : int  index in history_5d  (-1 if ground)
      s2_idx        : int  index in history_5d  (-1 if one-support or ground)
      projection    : list[float] × 4  [alpha_A, perp_A, alpha_B, perp_B]
    """
    if target_layer_id == 0:
        return 0, -1, -1, [0.0, 0.0, 0.0, 0.0]

    sup_layer  = target_layer_id - 1
    sup_indices = [i for i, lid in enumerate(history_layer_ids) if lid == sup_layer]
    if not sup_indices:
        return 0, -1, -1, [0.0, 0.0, 0.0, 0.0]

    tA, tB, _ = get_critical_points(target_pose)

    def nearest_support(tP):
        best_i, best_perp = -1, float('inf')
        for i in sup_indices:
            _, perp = project_crit_to_support(tP, history_5d[i])
            if abs(perp) < best_perp:
                best_perp, best_i = abs(perp), i
        return best_i

    s1_for_A = nearest_support(tA)
    s1_for_B = nearest_support(tB)

    if s1_for_A == s1_for_B:
        s1 = s1_for_A
        alpha_A, perp_A = project_crit_to_support(tA, history_5d[s1])
        alpha_B, perp_B = project_crit_to_support(tB, history_5d[s1])
        return 1, s1, -1, [alpha_A, perp_A, alpha_B, perp_B]
    else:
        s1, s2 = s1_for_A, s1_for_B
        alpha_A, perp_A = project_crit_to_support(tA, history_5d[s1])
        alpha_B, perp_B = project_crit_to_support(tB, history_5d[s2])
        return 2, s1, s2, [alpha_A, perp_A, alpha_B, perp_B]


# ── 17-dim brick token ─────────────────────────────────────────────────────────

def encode_brick(pose_5d, layer_id, time_index,
                 context_poses, context_layer_ids,
                 norm_stats, max_layer=MAX_LAYER_NORM):
    """
    17-dim feature vector for one brick token.

    Feature layout:
      base pose     (7): x_n, y_n, b, sin_r, cos_r, layer_norm, time_norm
      layer state   (3): rel_to_top, is_top, is_second_top
      same-layer    (5): count_norm, ndx, ndy, ndist, is_frontier
      geometry      (1): crit_span_norm  (half_span / std_xy)
      support ctx   (1): num_support_layer_norm  (bricks in layer-1 / 10)
    """
    ns = norm_stats
    x, y, z, b, r = float(pose_5d[0]), float(pose_5d[1]), float(pose_5d[2]), int(pose_5d[3]), float(pose_5d[4])
    r_c   = canonicalize_r(r)
    std_xy = (ns["std_x"] + ns["std_y"]) / 2.0

    max_ctx_lid = max(context_layer_ids, default=layer_id)
    max_ctx_lid = max(max_ctx_lid, layer_id)

    # Base pose (7)
    feat = [
        (x - ns["mean_x"]) / ns["std_x"],
        (y - ns["mean_y"]) / ns["std_y"],
        float(b), math.sin(r_c), math.cos(r_c),
        layer_id / max_layer,
        time_index / MAX_BRICKS,
    ]
    # Layer state (3)
    feat += [
        (max_ctx_lid - layer_id) / max(max_ctx_lid, 1),
        float(layer_id == max_ctx_lid),
        float(layer_id == max_ctx_lid - 1),
    ]
    # Same-layer occupancy (5)
    same = [context_poses[i] for i in range(len(context_poses))
            if context_layer_ids[i] == layer_id]
    if same:
        dxdy = [(p[0]-x, p[1]-y, math.sqrt((p[0]-x)**2 + (p[1]-y)**2)) for p in same]
        ndx, ndy, ndist = min(dxdy, key=lambda d: d[2])
        is_frontier = float(len(same) <= 2)
    else:
        ndx = ndy = ndist = 0.0
        is_frontier = 1.0
    feat += [len(same) / 12.0, ndx / ns["std_x"], ndy / ns["std_y"],
             ndist / std_xy, is_frontier]
    # Geometry (1)
    half_span = (BRICK_W if b == 0 else BRICK_D) / 2.0
    feat.append(half_span / std_xy)
    # Support context (1)
    num_sup = sum(1 for lid in context_layer_ids if lid == layer_id - 1)
    feat.append(min(num_sup, 10) / 10.0)

    assert len(feat) == FEATURE_DIM, f"Expected {FEATURE_DIM}, got {len(feat)}"
    return feat

### Compute layer vocabulary size

We look at all loaded sequences to set the classification head output size.  
This is an architectural capacity parameter, not a learned value.

In [ ]:
# MAX_LAYER_CLASSES: maximum layer_id + 1, scanned over all sequences
_max_lid = max(
    lid
    for seq in sequences
    for lid in assign_layer_ids(seq["poses"])
)
MAX_LAYER_CLASSES = _max_lid + 1
print(f"Maximum layer ID seen across all sequences: {_max_lid}")
print(f"Using MAX_LAYER_CLASSES = {MAX_LAYER_CLASSES}")

## 3. Sequence -> Next-Step Training Pairs

Key change: target is now `(x, y, sin_r, cos_r)` + discrete `b` + discrete `layer_id`.  
z is stored for reference but is **not** a prediction target.

In [ ]:
def sequence_to_pairs(seq_data):
    """
    Convert one sequence to T-1 next-step pairs with SE(2)-invariant support labels.

    Starts at t=1 so every pair has non-empty history.
    All support fields (support_state, s1_idx, s2_idx, projection) are
    SE(2)-invariant — augmentation only needs to transform history_raw.
    """
    poses     = seq_data["poses"]
    layer_ids = assign_layer_ids(poses)
    pairs = []
    for t in range(1, len(poses)):
        history_5d = [list(poses[i]) for i in range(t)]
        hist_lids  = [layer_ids[i]   for i in range(t)]
        target     = poses[t]
        tgt_layer  = layer_ids[t]

        support_state, s1_idx, s2_idx, projection = compute_support_labels(
            target, history_5d, hist_lids, tgt_layer,
        )

        pairs.append({
            "history_raw":       history_5d,
            "history_layer_ids": hist_lids,
            "target_b":          int(target[3]),
            "target_layer":      tgt_layer,
            "target_z":          float(target[2]),
            "support_state":     support_state,
            "s1_idx":            s1_idx,
            "s2_idx":            s2_idx,
            "projection":        projection,
            "seq_id":            seq_data["id"],
        })
    return pairs

## 4. Sequence-Level Train / Val / Test Split

In [ ]:
indices = list(range(len(sequences)))
random.shuffle(indices)

n_total = len(sequences)
n_val   = max(1, round(0.15 * n_total))
n_test  = max(1, round(0.15 * n_total))
n_train = n_total - n_val - n_test

train_idx = indices[:n_train]
val_idx   = indices[n_train : n_train + n_val]
test_idx  = indices[n_train + n_val :]

train_seqs = [sequences[i] for i in train_idx]
val_seqs   = [sequences[i] for i in val_idx]
test_seqs  = [sequences[i] for i in test_idx]

print(f"Split: {n_train} train / {n_val} val / {n_test} test  (total {n_total})")
print(f"Train: {[Path(sequences[i]['id']).name for i in train_idx]}")
print(f"Val:   {[Path(sequences[i]['id']).name for i in val_idx]}")
print(f"Test:  {[Path(sequences[i]['id']).name for i in test_idx]}")

train_pairs_orig = [p for seq in train_seqs for p in sequence_to_pairs(seq)]
val_pairs        = [p for seq in val_seqs   for p in sequence_to_pairs(seq)]
test_pairs       = [p for seq in test_seqs  for p in sequence_to_pairs(seq)]

print(f"\nPairs before aug -- train: {len(train_pairs_orig)}, val: {len(val_pairs)}, test: {len(test_pairs)}")

### Build z-Lookup Table (training data only)

Maps `layer_id -> mean z` so we can snap z after sampling.  
Built from training sequences **only** to respect the data split.

In [ ]:
def build_z_lookup(seqs):
    """Map layer_id -> mean z across given sequences (training only)."""
    acc = {}
    for seq in seqs:
        poses     = seq["poses"]
        layer_ids = assign_layer_ids(poses)
        for p, lid in zip(poses, layer_ids):
            acc.setdefault(lid, []).append(p[2])
    return {lid: float(np.mean(zs)) for lid, zs in sorted(acc.items())}


z_lookup = build_z_lookup(train_seqs)
print("z_lookup (layer_id -> mean z from training data):")
for lid, z_val in z_lookup.items():
    print(f"  layer {lid:2d}: z = {z_val:.6f} m")

# Safe z-snap: if predicted layer_id is out of lookup, use the last known z
def snap_z(layer_id):
    if layer_id in z_lookup:
        return z_lookup[layer_id]
    return z_lookup[max(z_lookup.keys())]  # extrapolate upward

## 5. SE(2) Data Augmentation  (training only)

Augmentation applies only to x, y, and rotation — **z is unchanged**,  
so the layer_id and z reference in the target remain correct after augmentation.

In [ ]:
def _rotate_xy(x, y, cx, cy, cos_t, sin_t, tx, ty):
    dx, dy = x - cx, y - cy
    return cx + cos_t * dx - sin_t * dy + tx, cy + sin_t * dx + cos_t * dy + ty


def se2_augment_pair(pair, theta, tx, ty):
    """
    Apply SE(2) to history_raw only.

    support_state, s1_idx, s2_idx, projection are SE(2)-invariant.
    history_layer_ids and target_* fields are also unchanged.
    """
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    hist = pair["history_raw"]

    if not hist:
        return pair

    cx = sum(h[0] for h in hist) / len(hist)
    cy = sum(h[1] for h in hist) / len(hist)

    new_history = []
    for h in hist:
        x, y, z, b, r = h
        xn, yn = _rotate_xy(x, y, cx, cy, cos_t, sin_t, tx, ty)
        new_history.append([xn, yn, z, b, canonicalize_r(r + theta)])

    return {**pair, "history_raw": new_history}


def augment_pairs(pairs, n_aug):
    aug = []
    for pair in pairs:
        for _ in range(n_aug):
            aug.append(se2_augment_pair(
                pair,
                theta=random.uniform(0, 2 * math.pi),
                tx=random.uniform(-0.01, 0.01),
                ty=random.uniform(-0.01, 0.01),
            ))
    return aug


print(f"Generating {N_AUG} augmentations × {len(train_pairs_orig)} training pairs...")
train_pairs_aug = augment_pairs(train_pairs_orig, N_AUG)
all_train_pairs = train_pairs_orig + train_pairs_aug
print(f"Total training pairs after augmentation: {len(all_train_pairs):,}")

## 6. Normalization Statistics  (training data only)

x, y, z normalisation is used for **input tokens** (z still context for the encoder).  
Only x, y normalisation is used for **target denormalisation** at inference.

In [ ]:
def compute_norm_stats(seqs):
    """Collect x, y, z statistics directly from raw sequence poses."""
    xs, ys, zs = [], [], []
    for seq in seqs:
        for p in seq["poses"]:
            xs.append(p[0]); ys.append(p[1]); zs.append(p[2])
    return {
        "mean_x": float(np.mean(xs)), "std_x": float(np.std(xs)) + 1e-8,
        "mean_y": float(np.mean(ys)), "std_y": float(np.std(ys)) + 1e-8,
        "mean_z": float(np.mean(zs)), "std_z": float(np.std(zs)) + 1e-8,
    }


norm_stats = compute_norm_stats(train_seqs)
print("Normalization statistics (from training sequences only):")
for k, v in norm_stats.items():
    print(f"  {k:8s}: {v:.6f}")

## 7. PyTorch Dataset & DataLoaders

Each sample returns **9** tensors:
```
tokens                 (MAX_BRICKS, FEATURE_DIM)   17-dim feature tokens
mask                   (MAX_BRICKS,)               bool: True = real brick
support_candidate_mask (MAX_BRICKS,)               bool: True = support-layer brick
target_b               scalar long                 0 (laying) or 1 (standing)
target_layer           scalar long                 discrete layer class
target_ss              scalar long                 support state 0/1/2
target_s1              scalar long                 s1 token index (MAX_BRICKS if absent)
target_s2              scalar long                 s2 token index (MAX_BRICKS if absent)
target_proj            (4,) float                  [alpha_A, perp_A, alpha_B, perp_B]
```

All label fields are **SE(2)-invariant** — augmentation only transforms `history_raw`.  
`encode_brick` is called lazily inside `__getitem__` after augmentation.

In [ ]:
class BrickPoseDataset(Dataset):
    """
    Returns 9 tensors per sample (see sec07 for layout).

    target_s1 / target_s2 = MAX_BRICKS when absent (used as ignore_index in CE loss).
    support_candidate_mask marks tokens whose layer_id == target_layer - 1.
    """

    def __init__(self, pairs, norm_stats, max_bricks=MAX_BRICKS):
        self.pairs      = pairs
        self.ns         = norm_stats
        self.max_bricks = max_bricks

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair      = self.pairs[idx]
        hist_raw  = pair["history_raw"]
        hist_lids = pair["history_layer_ids"]
        tgt_layer = pair["target_layer"]
        L         = self.max_bricks

        # ── Tokens ────────────────────────────────────────────────────────────
        tokens   = torch.zeros(L, FEATURE_DIM, dtype=torch.float32)
        mask     = torch.zeros(L, dtype=torch.bool)
        sup_mask = torch.zeros(L, dtype=torch.bool)

        for i, (pose, lid) in enumerate(zip(hist_raw[:L], hist_lids[:L])):
            tokens[i] = torch.tensor(
                encode_brick(pose, lid, i, hist_raw[:i], hist_lids[:i], self.ns),
                dtype=torch.float32,
            )
            mask[i] = True
            if lid == tgt_layer - 1:
                sup_mask[i] = True

        # ── Labels ────────────────────────────────────────────────────────────
        s1 = pair["s1_idx"]
        s2 = pair["s2_idx"]
        # Absent → sentinel MAX_BRICKS (used as ignore_index in CE loss)
        s1_t = min(s1, L) if s1 >= 0 else L
        s2_t = min(s2, L) if s2 >= 0 else L

        return (
            tokens,
            mask,
            sup_mask,
            torch.tensor(pair["target_b"],      dtype=torch.long),
            torch.tensor(pair["target_layer"],  dtype=torch.long),
            torch.tensor(pair["support_state"], dtype=torch.long),
            torch.tensor(s1_t,                 dtype=torch.long),
            torch.tensor(s2_t,                 dtype=torch.long),
            torch.tensor(pair["projection"],    dtype=torch.float32),
        )


train_ds = BrickPoseDataset(all_train_pairs, norm_stats)
val_ds   = BrickPoseDataset(val_pairs,       norm_stats)
test_ds  = BrickPoseDataset(test_pairs,      norm_stats)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_ds):,} samples  ({len(train_loader)} batches)")
print(f"Val:   {len(val_ds):,} samples  ({len(val_loader)} batches)")
print(f"Test:  {len(test_ds):,} samples  ({len(test_loader)} batches)")

## 8. Model Architecture (COG-Aware Multi-Head)

```
tokens → input_proj → Transformer encoder → scene (CLS)  + per-token embeddings
                                                  |
              ┌────────────────┬─────────────────┴──────────────────────┐
              ↓                ↓                                         ↓
           b_head          layer_head                               ss_head
           (2-class)       (MAX_LAYER_CLASSES)                    (3: 0/1/2)
              |                |
              └──── cond scalars (b, layer_norm) ────────────────────────┐
                                                                         ↓
                         s1_score_head([token | scene | b | layer]) → argmax over sup_mask
                                                                         ↓ s1_emb
                         s2_score_head([token | scene | b | layer | s1_emb]) → argmax
                                                                         ↓ s2_emb
                         proj_head([scene | s1_emb | s2_emb | b | layer]) → [alpha_A, perp_A, alpha_B, perp_B]
```

**Teacher forcing during training:** ground-truth `b`, `layer_id`, `s1_idx`, `s2_idx`.  
**Inference:** 3-pass — predict b/layer/ss → select s1/s2 → predict projection mean + noise.

In [ ]:
class NextBrickModel(nn.Module):
    """
    COG-aware staged prediction model.

    Heads:
      b_head      scene → binary logits (2)
      layer_head  scene → layer logits (MAX_LAYER_CLASSES)
      ss_head     scene → support-state logits (N_SUPPORT_STATES=3)
      s1_score_head  [token|scene|b|layer] → score per token
      s2_score_head  [token|scene|b|layer|s1_emb] → score per token
      proj_head   [scene|s1_emb|s2_emb|b|layer] → PROJ_DIM=4
    """

    def __init__(
        self,
        feature_dim=FEATURE_DIM,
        hidden_dim=HIDDEN_DIM,
        nhead=N_HEADS,
        num_layers=N_LAYERS,
        ff_dim=FF_DIM,
        dropout=DROPOUT,
        max_layer_classes=None,
    ):
        super().__init__()
        H = hidden_dim
        n_layer_cls = max_layer_classes or MAX_LAYER_CLASSES
        self.hidden_dim = H

        self.input_proj = nn.Sequential(
            nn.Linear(feature_dim, H), nn.ReLU(), nn.Linear(H, H),
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, H) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=H, nhead=nhead, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # Scene-level classification heads
        self.b_head = nn.Sequential(
            nn.Linear(H, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 2),
        )
        self.layer_head = nn.Sequential(
            nn.Linear(H, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, n_layer_cls),
        )
        self.ss_head = nn.Sequential(
            nn.Linear(H, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, N_SUPPORT_STATES),
        )

        # Per-token selection heads
        # s1: [token(H) | scene(H) | b(1) | layer_norm(1)]
        self.s1_score_head = nn.Linear(2 * H + 2, 1)
        # s2: additionally conditioned on s1_emb
        self.s2_score_head = nn.Linear(3 * H + 2, 1)
        # Learnable embedding for absent support (used when s1/s2 index = -1)
        self.no_support_emb = nn.Parameter(torch.randn(H) * 0.02)

        # Projection head: [scene(H) | s1_emb(H) | s2_emb(H) | b(1) | layer_norm(1)]
        self.proj_head = nn.Sequential(
            nn.Linear(3 * H + 2, 128), nn.ReLU(), nn.Linear(128, PROJ_DIM),
        )

    def _encode(self, tokens, mask):
        """Shared encoder: returns (scene, token_embs)."""
        B = tokens.shape[0]
        x   = self.input_proj(tokens)
        cls = self.cls_token.expand(B, 1, -1)
        x   = torch.cat([cls, x], dim=1)
        cls_valid = torch.ones(B, 1, dtype=torch.bool, device=mask.device)
        x   = self.transformer(x, src_key_padding_mask=~torch.cat([cls_valid, mask], dim=1))
        return x[:, 0], x[:, 1:]   # scene (B,H), token_embs (B,L,H)

    def _gather_support_emb(self, token_embs, idx):
        """Gather per-batch token embedding; return no_support_emb where idx >= L."""
        B, L, H = token_embs.shape
        absent  = idx >= L
        safe    = idx.clamp(0, L - 1)
        emb     = token_embs[torch.arange(B, device=token_embs.device), safe]
        no_emb  = self.no_support_emb.unsqueeze(0).expand(B, -1)
        emb[absent] = no_emb[absent]
        return emb

    def forward(
        self,
        tokens,          # (B, L, FEATURE_DIM)
        mask,            # (B, L)  True = real token
        sup_mask,        # (B, L)  True = support-layer candidate
        cond_b=None,     # (B,) long  — teacher forcing
        cond_layer=None, # (B,) long  — teacher forcing
        cond_s1=None,    # (B,) long  index or >= L means absent
        cond_s2=None,    # (B,) long
    ):
        B, L, _ = tokens.shape
        scene, token_embs = self._encode(tokens, mask)

        b_logits     = self.b_head(scene)
        layer_logits = self.layer_head(scene)
        ss_logits    = self.ss_head(scene)

        # Conditioning scalars broadcast over sequence length
        b_scalar = (cond_b.float() if cond_b is not None
                    else b_logits.argmax(-1).float()).unsqueeze(-1)         # (B,1)
        layer_scalar = ((cond_layer.float() if cond_layer is not None
                         else layer_logits.argmax(-1).float())
                        / max(MAX_LAYER_CLASSES - 1, 1)).unsqueeze(-1)     # (B,1)

        scene_bc = scene.unsqueeze(1).expand(-1, L, -1)                    # (B,L,H)
        b_bc     = b_scalar.unsqueeze(1).expand(-1, L, -1)                 # (B,L,1)
        lyr_bc   = layer_scalar.unsqueeze(1).expand(-1, L, -1)             # (B,L,1)

        # ── s1 scoring ────────────────────────────────────────────────────────
        s1_inp    = torch.cat([token_embs, scene_bc, b_bc, lyr_bc], dim=-1)  # (B,L,2H+2)
        s1_logits = self.s1_score_head(s1_inp).squeeze(-1)                   # (B,L)
        s1_logits = s1_logits.masked_fill(~sup_mask, float('-inf'))
        s1_logits = s1_logits.masked_fill(~mask,     float('-inf'))

        # Gather s1 embedding (teacher-forced or predicted)
        s1_idx = cond_s1 if cond_s1 is not None else s1_logits.argmax(-1)
        s1_emb = self._gather_support_emb(token_embs, s1_idx)               # (B,H)

        # ── s2 scoring ────────────────────────────────────────────────────────
        s1_emb_bc = s1_emb.unsqueeze(1).expand(-1, L, -1)                    # (B,L,H)
        s2_inp    = torch.cat([token_embs, scene_bc, b_bc, lyr_bc, s1_emb_bc], dim=-1)
        s2_logits = self.s2_score_head(s2_inp).squeeze(-1)                   # (B,L)
        s2_logits = s2_logits.masked_fill(~sup_mask, float('-inf'))
        s2_logits = s2_logits.masked_fill(~mask,     float('-inf'))

        s2_idx = cond_s2 if cond_s2 is not None else s2_logits.argmax(-1)
        s2_emb = self._gather_support_emb(token_embs, s2_idx)               # (B,H)

        # ── Projection ────────────────────────────────────────────────────────
        proj_inp = torch.cat([scene, s1_emb, s2_emb, b_scalar, layer_scalar], dim=-1)
        proj     = self.proj_head(proj_inp)                                  # (B,4)

        return b_logits, layer_logits, ss_logits, s1_logits, s2_logits, proj

## 9. Loss Functions

In [ ]:
def compute_loss(
    b_logits, layer_logits, ss_logits, s1_logits, s2_logits, proj,
    target_b, target_layer, target_ss, target_s1, target_s2, target_proj,
    max_bricks=MAX_BRICKS,
):
    """
    Multi-task loss.

    s1/s2 CE use ignore_index=max_bricks so absent (ground-layer) samples
    contribute zero loss — the sentinel value equals the absent-token index.
    """
    l_b     = F.cross_entropy(b_logits,     target_b)
    l_layer = F.cross_entropy(layer_logits, target_layer)
    l_ss    = F.cross_entropy(ss_logits,    target_ss)

    # s1_logits: (B, L)  target_s1: (B,)  ignore absent targets (= max_bricks)
    l_s1 = F.cross_entropy(s1_logits, target_s1, ignore_index=max_bricks)
    l_s2 = F.cross_entropy(s2_logits, target_s2, ignore_index=max_bricks)

    l_proj = F.smooth_l1_loss(proj, target_proj)

    total = (LAMBDA_B     * l_b     +
             LAMBDA_LAYER * l_layer +
             LAMBDA_SS    * l_ss    +
             LAMBDA_S1    * l_s1    +
             LAMBDA_S2    * l_s2    +
             LAMBDA_PROJ  * l_proj)
    return total, l_b, l_layer, l_ss, l_s1, l_s2, l_proj

## 10. Training & Validation Loops

Teacher forcing: ground-truth `b` and `layer_id` are passed to the MDN during **both** train and val.  
This gives a fair measure of how well the MDN learns the conditional pose distribution.

In [ ]:
def train_epoch(model, loader, optimizer):
    model.train()
    total = 0.0
    for batch in loader:
        (tokens, mask, sup_mask,
         tgt_b, tgt_layer, tgt_ss, tgt_s1, tgt_s2, tgt_proj) = [x.to(device) for x in batch]
        optimizer.zero_grad()
        out = model(tokens, mask, sup_mask,
                    cond_b=tgt_b, cond_layer=tgt_layer,
                    cond_s1=tgt_s1, cond_s2=tgt_s2)
        loss, *_ = compute_loss(*out, tgt_b, tgt_layer, tgt_ss, tgt_s1, tgt_s2, tgt_proj)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    run = {k: 0.0 for k in ["tot", "b", "layer", "ss", "s1", "s2", "proj"]}
    corr_b = corr_layer = corr_ss = n = 0
    for batch in loader:
        (tokens, mask, sup_mask,
         tgt_b, tgt_layer, tgt_ss, tgt_s1, tgt_s2, tgt_proj) = [x.to(device) for x in batch]
        out = model(tokens, mask, sup_mask,
                    cond_b=tgt_b, cond_layer=tgt_layer,
                    cond_s1=tgt_s1, cond_s2=tgt_s2)
        b_logits, layer_logits, ss_logits, _, _, _ = out
        loss, l_b, l_layer, l_ss, l_s1, l_s2, l_proj = compute_loss(
            *out, tgt_b, tgt_layer, tgt_ss, tgt_s1, tgt_s2, tgt_proj,
        )
        run["tot"]   += loss.item();   run["b"]    += l_b.item()
        run["layer"] += l_layer.item(); run["ss"]  += l_ss.item()
        run["s1"]    += l_s1.item();   run["s2"]   += l_s2.item()
        run["proj"]  += l_proj.item()
        corr_b     += (b_logits.argmax(-1)     == tgt_b).sum().item()
        corr_layer += (layer_logits.argmax(-1) == tgt_layer).sum().item()
        corr_ss    += (ss_logits.argmax(-1)    == tgt_ss).sum().item()
        n += len(tgt_b)
    N = len(loader)
    return (run["tot"]/N, run["b"]/N, run["layer"]/N,
            run["ss"]/N,  run["s1"]/N, run["s2"]/N, run["proj"]/N,
            corr_b/n, corr_layer/n, corr_ss/n)

## 11. Main Training Loop

In [ ]:
model = NextBrickModel(
    feature_dim=FEATURE_DIM,
    hidden_dim=HIDDEN_DIM,
    ff_dim=FF_DIM,
    max_layer_classes=MAX_LAYER_CLASSES,
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10,
)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")
print(f"FEATURE_DIM={FEATURE_DIM}  HIDDEN_DIM={HIDDEN_DIM}  FF_DIM={FF_DIM}  PROJ_DIM={PROJ_DIM}")

best_val, no_improve = float("inf"), 0
CKPT = Path("training_data/trained_models/best_model_cog_aware.pth")
CKPT.parent.mkdir(parents=True, exist_ok=True)

log = {k: [] for k in [
    "train", "val", "val_b", "val_layer", "val_ss",
    "val_s1", "val_s2", "val_proj",
    "b_acc", "layer_acc", "ss_acc",
]}

for epoch in range(1, MAX_EPOCHS + 1):
    tr = train_epoch(model, train_loader, optimizer)
    (vl, vl_b, vl_layer, vl_ss, vl_s1, vl_s2, vl_proj,
     b_acc, layer_acc, ss_acc) = eval_epoch(model, val_loader)
    scheduler.step(vl)

    log["train"].append(tr);         log["val"].append(vl)
    log["val_b"].append(vl_b);       log["val_layer"].append(vl_layer)
    log["val_ss"].append(vl_ss);     log["val_s1"].append(vl_s1)
    log["val_s2"].append(vl_s2);     log["val_proj"].append(vl_proj)
    log["b_acc"].append(b_acc);      log["layer_acc"].append(layer_acc)
    log["ss_acc"].append(ss_acc)

    if vl < best_val:
        best_val, no_improve = vl, 0
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "norm_stats": norm_stats,
            "z_lookup": z_lookup,
            "max_layer_classes": MAX_LAYER_CLASSES,
            "feature_dim": FEATURE_DIM,
            "hidden_dim": HIDDEN_DIM,
            "val_loss": vl,
        }, CKPT)
    else:
        no_improve += 1

    if epoch % 10 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]["lr"]
        print(
            f"[{epoch:3d}] train={tr:.3f} | val={vl:.3f} "
            f"(b={vl_b:.3f} lyr={vl_layer:.3f} ss={vl_ss:.3f} "
            f"s1={vl_s1:.3f} s2={vl_s2:.3f} proj={vl_proj:.3f}) "
            f"| acc: b={b_acc:.3f} lyr={layer_acc:.3f} ss={ss_acc:.3f} "
            f"| lr={lr_now:.1e}"
        )
    if no_improve >= PATIENCE:
        print(f"\nEarly stop at epoch {epoch}.")
        break

print(f"\nBest val loss: {best_val:.4f}  ->  {CKPT}")

## 12. Training Curves

In [ ]:
E = range(1, len(log["train"]) + 1)
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

axes[0,0].plot(E, log["train"], label="train"); axes[0,0].plot(E, log["val"], label="val")
axes[0,0].set_title("Total Loss"); axes[0,0].set_xlabel("Epoch"); axes[0,0].legend()

axes[0,1].plot(E, log["val_b"],     label="b CE")
axes[0,1].plot(E, log["val_layer"], label="layer CE")
axes[0,1].plot(E, log["val_ss"],    label="ss CE")
axes[0,1].set_title("Val — Discrete Losses"); axes[0,1].set_xlabel("Epoch"); axes[0,1].legend()

axes[0,2].plot(E, log["val_s1"], label="s1 CE")
axes[0,2].plot(E, log["val_s2"], label="s2 CE")
axes[0,2].plot(E, log["val_proj"], label="proj SmoothL1")
axes[0,2].set_title("Val — Selection & Projection Losses")
axes[0,2].set_xlabel("Epoch"); axes[0,2].legend()

axes[1,0].plot(E, log["b_acc"])
axes[1,0].set_title("Val Binary-State Accuracy (b)")
axes[1,0].set_xlabel("Epoch"); axes[1,0].set_ylim(0, 1)
axes[1,0].axhline(0.5, color="gray", linestyle="--", alpha=0.5)

axes[1,1].plot(E, log["layer_acc"])
axes[1,1].set_title("Val Layer-ID Accuracy")
axes[1,1].set_xlabel("Epoch"); axes[1,1].set_ylim(0, 1)
axes[1,1].axhline(1 / MAX_LAYER_CLASSES, color="gray", linestyle="--", alpha=0.5, label="chance")
axes[1,1].legend()

axes[1,2].plot(E, log["ss_acc"])
axes[1,2].set_title("Val Support-State Accuracy")
axes[1,2].set_xlabel("Epoch"); axes[1,2].set_ylim(0, 1)
axes[1,2].axhline(1 / N_SUPPORT_STATES, color="gray", linestyle="--", alpha=0.5, label="chance")
axes[1,2].legend()

plt.tight_layout()
plt.savefig("training_data/training_curves_cog_aware.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_data/training_curves_cog_aware.png")

## 13. Inference — 3-Pass Staged Sampling

**Pipeline:**
1. **Pass 1:** Encode history → predict `b`, `layer_id`, `support_state`
2. **Pass 2:** Given `(b, layer_id)`, score each history token → argmax over sup_mask → `s1`, `s2`
3. **Pass 3:** Conditioned on `s1`, `s2` → predict projection mean → add Gaussian noise from `SIGMA_PROJ_INFER`
4. Decode `(x, y, r)` from `[alpha_A, perp_A, alpha_B, perp_B]` onto support critical lines
5. Snap `z` from `z_lookup[layer_id]`
6. Assemble full 5D pose candidate `[x, y, z, b, r]`

In [ ]:
def decode_pose_from_projections(support_state, s1_pose, s2_pose,
                                  alpha_A, perp_A, alpha_B, perp_B):
    """
    Recover world (x, y, r) from critical-point projections.

    s1_pose / s2_pose : [x, y, z, b, r]
    Returns [x, y, r] or None if support_state == 0.
    """
    def decode_point(sup_pose, alpha, perp):
        sA, sB, s_half = get_critical_points(sup_pose)
        span = 2.0 * s_half
        dx, dy = sB[0] - sA[0], sB[1] - sA[1]
        d  = math.sqrt(dx * dx + dy * dy + 1e-8)
        ax, ay = dx / d, dy / d
        px, py = -ay, ax
        wx = sA[0] + alpha * span * ax + perp * span * px
        wy = sA[1] + alpha * span * ay + perp * span * py
        return [wx, wy]

    if support_state == 0 or s1_pose is None:
        return None

    tA = decode_point(s1_pose, alpha_A, perp_A)
    tB = decode_point(s2_pose if (support_state == 2 and s2_pose is not None) else s1_pose,
                      alpha_B, perp_B)

    cx = (tA[0] + tB[0]) / 2.0
    cy = (tA[1] + tB[1]) / 2.0
    r  = canonicalize_r(math.atan2(tA[1] - tB[1], tA[0] - tB[0]))
    return [cx, cy, r]


def sample_next_brick(model, history_5d, norm_stats, z_lookup, n_candidates=100):
    """
    3-pass staged inference for the COG-aware model.

    Returns (candidates, pred_b, pred_layer, pred_ss) where candidates are
    dicts {x, y, z, b, r, sin_r, cos_r, layer_id, support_state, s1_idx, s2_idx}.
    """
    ns = norm_stats
    model.eval()

    layer_ids = assign_layer_ids(history_5d) if history_5d else []
    encoded = [
        encode_brick(history_5d[t], layer_ids[t], t,
                     history_5d[:t], layer_ids[:t], ns)
        for t in range(len(history_5d))
    ]

    tokens = torch.zeros(1, MAX_BRICKS, FEATURE_DIM, dtype=torch.float32)
    mask   = torch.zeros(1, MAX_BRICKS, dtype=torch.bool)
    for i, h in enumerate(encoded[:MAX_BRICKS]):
        tokens[0, i] = torch.tensor(h, dtype=torch.float32)
        mask[0, i]   = True
    tokens = tokens.to(device)
    mask   = mask.to(device)

    # ── Pass 1: discrete predictions ─────────────────────────────────────────
    dummy_sup = torch.zeros(1, MAX_BRICKS, dtype=torch.bool, device=device)
    with torch.no_grad():
        b_logits, layer_logits, ss_logits, _, _, _ = model(tokens, mask, dummy_sup)

    pred_b     = int(b_logits[0].argmax())
    pred_layer = int(layer_logits[0].argmax())
    pred_ss    = int(ss_logits[0].argmax())
    pred_z     = snap_z(pred_layer)

    # Build support candidate mask for the predicted layer
    sup_mask = torch.zeros(1, MAX_BRICKS, dtype=torch.bool, device=device)
    for i, lid in enumerate(layer_ids[:MAX_BRICKS]):
        if lid == pred_layer - 1:
            sup_mask[0, i] = True

    b_t     = torch.tensor([pred_b],     dtype=torch.long, device=device)
    layer_t = torch.tensor([pred_layer], dtype=torch.long, device=device)

    # ── Pass 2: select s1, s2 ────────────────────────────────────────────────
    with torch.no_grad():
        _, _, _, s1_logits, s2_logits, proj_mean = model(
            tokens, mask, sup_mask, cond_b=b_t, cond_layer=layer_t,
        )

    valid_sup = (sup_mask & mask)[0]

    s1_sc = s1_logits[0].clone()
    s1_sc[~valid_sup] = float('-inf')
    pred_s1 = int(s1_sc.argmax()) if valid_sup.any() else -1
    s1_pose = history_5d[pred_s1] if pred_s1 >= 0 else None

    # s2: exclude same token as s1 when multiple supports available
    s2_sc = s2_logits[0].clone()
    s2_sc[~valid_sup] = float('-inf')
    if valid_sup.sum() > 1:
        pred_s2 = int(s2_sc.argmax())
        s2_pose = history_5d[pred_s2]
    else:
        pred_s2 = -1
        s2_pose = None

    # ── Pass 3: sample projections ────────────────────────────────────────────
    mu_proj = proj_mean[0].cpu().numpy()   # (4,)
    sigmas  = np.array(SIGMA_PROJ_INFER)

    candidates = []
    for _ in range(n_candidates):
        proj_s = mu_proj + np.random.randn(4) * sigmas
        alpha_A, perp_A, alpha_B, perp_B = proj_s.tolist()

        if pred_ss == 0 or s1_pose is None:
            # Ground layer: no projections — sample near existing centroid
            lx = [h[0] for h in history_5d] or [0.0]
            ly = [h[1] for h in history_5d] or [0.0]
            x  = float(np.mean(lx)) + random.gauss(0, 0.02)
            y  = float(np.mean(ly)) + random.gauss(0, 0.02)
            r  = canonicalize_r(random.uniform(0, math.pi))
        else:
            result = decode_pose_from_projections(
                pred_ss, s1_pose, s2_pose, alpha_A, perp_A, alpha_B, perp_B,
            )
            if result is None:
                continue
            x, y, r = result

        candidates.append({
            "x": x, "y": y, "z": pred_z,
            "b": pred_b, "r": r,
            "sin_r": math.sin(r), "cos_r": math.cos(r),
            "layer_id": pred_layer,
            "support_state": pred_ss,
            "s1_idx": pred_s1, "s2_idx": pred_s2,
        })

    return candidates, pred_b, pred_layer, pred_ss

In [ ]:
# Load best checkpoint
ckpt = torch.load(CKPT, map_location=device)
model.load_state_dict(ckpt["model_state"])
print(f"Loaded checkpoint: epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")

# Pick a val pair with a long history for a meaningful demo
demo_pair = max(val_pairs, key=lambda p: len(p["history_raw"]))
print(f"Demo seq: {Path(demo_pair['seq_id']).name}  |  "
      f"history: {len(demo_pair['history_raw'])} bricks")

history_5d = [list(h) for h in demo_pair["history_raw"]]
candidates, pred_b, pred_layer, pred_ss = sample_next_brick(
    model, history_5d, norm_stats, z_lookup, n_candidates=100,
)
pred_z = snap_z(pred_layer)
ss_names = {0: "ground", 1: "one-support", 2: "two-support"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
hx = [h[0] for h in history_5d]
hy = [h[1] for h in history_5d]
ax.scatter(hx, hy, c="steelblue", s=40, alpha=0.7, label="History", zorder=3)

tgt = demo_pair
tgt_pose = demo_pair["history_raw"][-1]   # last history brick as proxy for layout
ax.scatter(
    [c["x"] for c in candidates],
    [c["y"] for c in candidates],
    c="tomato", s=20, alpha=0.5,
    label=f"Candidates (layer={pred_layer}, {ss_names[pred_ss]})", zorder=4,
)
if candidates[0]["s1_idx"] >= 0:
    s1 = history_5d[candidates[0]["s1_idx"]]
    ax.scatter([s1[0]], [s1[1]], c="gold", s=200, marker="D", label="s1 support", zorder=5)
if candidates[0]["s2_idx"] >= 0:
    s2 = history_5d[candidates[0]["s2_idx"]]
    ax.scatter([s2[0]], [s2[1]], c="orange", s=200, marker="D", label="s2 support", zorder=5)
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.set_title("Sampled Candidates — Top-Down View")
ax.legend(); ax.set_aspect("equal")

# Layer probability bar chart
ax2 = axes[1]
with torch.no_grad():
    tokens = torch.zeros(1, MAX_BRICKS, FEATURE_DIM, dtype=torch.float32)
    mask_t = torch.zeros(1, MAX_BRICKS, dtype=torch.bool)
    layer_ids_d = assign_layer_ids(history_5d)
    for i, (pose, lid) in enumerate(zip(history_5d[:MAX_BRICKS], layer_ids_d[:MAX_BRICKS])):
        tokens[0, i] = torch.tensor(
            encode_brick(pose, lid, i, history_5d[:i], layer_ids_d[:i], norm_stats),
            dtype=torch.float32)
        mask_t[0, i] = True
    dummy_sup = torch.zeros(1, MAX_BRICKS, dtype=torch.bool, device=device)
    b_logits, layer_logits, ss_logits, _, _, _ = model(
        tokens.to(device), mask_t.to(device), dummy_sup)
    b_prob     = F.softmax(b_logits[0],     dim=-1).cpu().numpy()
    layer_prob = F.softmax(layer_logits[0], dim=-1).cpu().numpy()
    ss_prob    = F.softmax(ss_logits[0],    dim=-1).cpu().numpy()

ax2.bar(range(len(layer_prob)), layer_prob, color="steelblue", alpha=0.7)
ax2.axvline(pred_layer, color="tomato", linewidth=2, linestyle="--",
            label=f"Pred layer={pred_layer} (z={pred_z:.4f}m)")
ax2.set_xlabel("Layer ID"); ax2.set_ylabel("P(layer)")
ax2.set_title(f"Layer Distribution  |  ss={ss_names[pred_ss]} [{ss_prob}]")
ax2.legend()

plt.tight_layout()
plt.savefig("training_data/candidate_visualization_cog_aware.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Predicted b: {'laying' if pred_b == 0 else 'standing'}  "
      f"(laying={b_prob[0]:.3f} standing={b_prob[1]:.3f})")
print(f"Predicted layer={pred_layer}, snapped z={pred_z:.4f}m")
print(f"Support state: {ss_names[pred_ss]}  "
      f"(ground={ss_prob[0]:.3f} one={ss_prob[1]:.3f} two={ss_prob[2]:.3f})")
print(f"Generated {len(candidates)} candidates")